<a href="https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahagngrigoryan2006/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**What one row means.** One content item, scored at the decision moment **2026-06-01**, using only
data observable by then (May and earlier). The label it is later checked against is whether that
item's impressions declined more than 20% in **June 2026** versus May — an outcome that had not
happened yet at the decision moment.

**Why the rule orders the queue.** Across 10 client-grouped folds on the April→May frame, I
observed the frozen rule has mean and weighted mean precision@50 noticably higher than the models (see the code's output). At the
review budget, the rule ordered the head of the queue at least as well as either model in this
data. That is an **observed** ranking difference on one frame, not evidence that the rule is
generally superior: `w06_validation_audit.ipynb` measured the three methods' *precision* as
statistically indistinguishable (95% CI on every pairwise difference includes zero), and the models
clearly beat the rule on **recall** (tree 0.766 vs rule 0.111) and therefore F1. The rule wins where
this playbook happens to operate — the top of a short list — and loses badly on total coverage.

**Why the models still appear on every row.** They carry different information than the ordering
does. The tree's classification and its decision path say *which combination of signals* put an item
where it is; the logistic regression's probability gives a graded second opinion. Where they
disagree with the rule, that disagreement is itself the useful signal — see `QUIET_RISK_MODEL_ONLY`
below.

**Reason codes on every row:**

| Column | Values | What it tells the reviewer |
|---|---|---|
| `rule_reason` | `CTR_BELOW_POSITION_PEERS` / `NO_RULE_FLAG` | The frozen rule fired: eligible, and zero clicks against a non-zero peer benchmark. |
| `lr_reason` | `LR_HIGH_RISK` / `LR_MODERATE` / `LR_LOW_RISK` | Banded logistic-regression probability (exact value in `lr_decline_probability`). |
| `tree_reason` | `TREE_FLAGS_DECLINE` / `TREE_NO_DECLINE` | The shallow tree's own call. |
| `tree_path` | e.g. `prev_30_ctr<=0.002 AND prev_30_impressions>386` | The ≤3 conditions that produced that call — the tree's actual reasoning, in full. |
| `agreement` | `RULE_AND_MODEL_AGREE` / `RULE_ONLY` / `QUIET_RISK_MODEL_ONLY` / `NO_FLAG` | Where rule and models diverge. |

**`QUIET_RISK_MODEL_ONLY` deserves its own line.** These are items a model flags but the rule does
not — usually because the rule's eligibility gates or its strict `ctr == 0` condition exclude them.
The model offers no reason a human can read beyond `tree_path`, so these are routed to human
judgement rather than straight onto the review list. The rule supplies the rhythm; the models
supply a second opinion; neither authorises an edit.

**The eligibility gates carried over unchanged from `w04_baseline_score.ipynb`:**
`prev_30_impressions >= 500` (enough traffic to judge a CTR), `content_age_days_at_decision >= 90`
(not brand-new), `days_since_last_update_at_decision >= 60` (not just optimised).

**One tier-mean caveat, carried from `w04`:** `expected_ctr_for_tier` is pooled across clients
within each position tier. `w06_validation_audit.ipynb` raised exactly this concern about the
research paper's own Finding #3 — a pooled ratio can be dominated by whichever client has the most
traffic in that tier. The same limitation applies here and is repeated in Section 2.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one -- typing sentences here breaks Run All.

import os, json
import numpy as np
import pandas as pd
import duckdb
import scipy.sparse as sp
import matplotlib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

REPO_URL = "https://github.com/vahagngrigoryan2006/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"
import subprocess
if not os.path.isdir(REPO_DIR) and os.path.basename(os.getcwd()) != REPO_DIR:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN as a Colab secret before continuing."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"read_parquet('{WAREHOUSE}/dim_content.parquet')"

# Read months 03-06 ONCE into a local table, then build both frames against it.
# The flyrank-data skill warns that repeated full scans hit HTTP 429 -- this is the single remote read.
MONTHS = ["2026-03", "2026-04", "2026-05", "2026-06"]
FACT_PARTS = ", ".join([f"'{WAREHOUSE}/fact_content_daily_performance/month={m}/*.parquet'" for m in MONTHS])
con.execute(f"CREATE OR REPLACE TABLE fact_4mo AS SELECT * FROM read_parquet([{FACT_PARTS}])")
print("fact_4mo rows:", con.sql("SELECT COUNT(*) FROM fact_4mo").fetchone()[0])
print("date span:", con.sql("SELECT MIN(report_date), MAX(report_date) FROM fact_4mo").fetchone())


def build_frame(as_of_date, decision_moment):
    """Same query shape as w05_model.ipynb, parameterised by as_of date.

    as_of 2026-05-31 -> prev_30 = April,      last_30 = May   (TRAINING frame)
    as_of 2026-07-01 -> prev_30 = May 2-31,   last_30 = June  (SEALED JUNE frame)
    """
    features = con.sql(f"""
        WITH bounds AS (SELECT DATE '{as_of_date}' AS as_of_date),
        per_item AS (
            SELECT f.client_hash_id, f.content_hash_id,
                   MIN(f.report_date) AS first_seen,
                   SUM(CASE WHEN f.report_date >= b.as_of_date - INTERVAL 30 DAY AND f.report_date <= b.as_of_date
                            THEN f.gsc_impressions ELSE 0 END) AS last_30_impressions,
                   SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.gsc_impressions ELSE 0 END) AS prev_30_impressions,
                   AVG(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.gsc_avg_position END) AS prev_30_avg_position,
                   SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.gsc_clicks ELSE 0 END) AS prev_30_clicks,
                   CASE WHEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                      THEN f.gsc_impressions ELSE 0 END) > 0
                        THEN SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                      THEN f.gsc_clicks ELSE 0 END)
                             / SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                                      THEN f.gsc_impressions ELSE 0 END)
                   END AS prev_30_ctr,
                   SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.ga4_sessions ELSE 0 END) AS prev_30_sessions,
                   SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.ga4_engaged_sessions ELSE 0 END) AS prev_30_engaged_sessions,
                   SUM(CASE WHEN f.report_date < b.as_of_date - INTERVAL 30 DAY AND f.report_date >= b.as_of_date - INTERVAL 60 DAY
                            THEN f.scroll_events ELSE 0 END) AS prev_30_scroll_events
            FROM fact_4mo f, bounds b
            GROUP BY 1, 2
        )
        SELECT p.*, d.content_created_date, d.content_updated_date, d.keyword_created_date,
               d.keyword_char_count, d.keyword_token_count, d.content_type, d.search_volume,
               d.competition, d.cpc, d.main_intent, d.backlinks, d.category_count,
               d.char_count, d.word_count
        FROM per_item p
        JOIN {DIM_CONTENT} d USING (content_hash_id)
        WHERE p.first_seen <= DATE '{as_of_date}' - INTERVAL 60 DAY   -- guard (a)
          AND p.prev_30_impressions >= 100                             -- guard (b)
    """).df()

    dm = pd.Timestamp(decision_moment)
    for c in ["content_created_date", "content_updated_date", "keyword_created_date"]:
        features[c] = pd.to_datetime(features[c])
    features["content_age_days_at_decision"] = (dm - features["content_created_date"]).dt.days
    features["days_since_last_update_at_decision"] = (dm - features["content_updated_date"]).dt.days
    features["keyword_age_days_at_decision"] = (dm - features["keyword_created_date"]).dt.days
    features = features[features["days_since_last_update_at_decision"] > 0]      # guard (c)

    features["impressions_pct_change"] = (
        (features["last_30_impressions"] - features["prev_30_impressions"]) / features["prev_30_impressions"]
    )
    features["is_declining"] = (features["impressions_pct_change"] < -0.20).astype(int)

    # Same 23-column feature set + NA handling as w03_feature_leakage_check.ipynb / w05 / w06
    keep = ["prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
            "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
            "content_age_days_at_decision", "days_since_last_update_at_decision",
            "keyword_char_count", "keyword_token_count", "content_type", "search_volume",
            "competition", "cpc", "keyword_age_days_at_decision", "main_intent", "backlinks",
            "category_count", "char_count", "word_count"]
    X = features[keep].copy()
    X["word_count_tier"] = pd.cut(X["word_count"], bins=[-np.inf, 1000, 2000, 3500, np.inf],
                                   labels=["<1000", "1000-2000", "2000-3500", "3500+"],
                                   right=False).astype("object").fillna("NA")
    X["char_count_tier"] = pd.cut(X["char_count"], bins=[-np.inf, 8000, 15000, 25000, np.inf],
                                   labels=["<8000", "8000-15000", "15000-25000", "25000+"],
                                   right=False).astype("object").fillna("NA")
    X = X.drop(columns=["word_count", "char_count"])
    X["no_keyword_data"] = X["competition"].isna().astype(int)
    for c in ["search_volume", "competition", "cpc"]:
        X[c] = X[c].fillna(0)
    X["keyword_age_days_at_decision"] = X["keyword_age_days_at_decision"].fillna(-30)
    X["main_intent"] = X["main_intent"].fillna("NA")
    X["backlinks_na"] = X["backlinks"].isna().astype(int)
    X["backlinks"] = X["backlinks"].fillna(0)

    meta = features[["client_hash_id", "content_hash_id", "is_declining"]].copy()
    return X.reset_index(drop=True), meta.reset_index(drop=True)


NUMERIC_COLS = ["prev_30_impressions", "prev_30_avg_position", "prev_30_clicks", "prev_30_ctr",
                "prev_30_sessions", "prev_30_engaged_sessions", "prev_30_scroll_events",
                "content_age_days_at_decision", "days_since_last_update_at_decision",
                "keyword_char_count", "keyword_token_count", "search_volume", "competition", "cpc",
                "keyword_age_days_at_decision", "backlinks", "category_count",
                "no_keyword_data", "backlinks_na"]
CATEGORICAL_COLS = ["content_type", "main_intent", "word_count_tier", "char_count_tier"]

# TRAINING frame: April -> May, identical to w05/w06
X_train, meta_train = build_frame("2026-05-31", "2026-05-01")
# SEALED JUNE frame: May -> June. Read once, here.
X_june, meta_june = build_frame("2026-07-01", "2026-06-01")

assert list(X_train.columns) == list(X_june.columns), "Frames must share the same feature schema."
assert set(NUMERIC_COLS + CATEGORICAL_COLS) == set(X_train.columns)
print(f"\nTraining frame (Apr->May): {len(X_train):,} rows, {meta_train['client_hash_id'].nunique()} clients, "
      f"base rate {meta_train['is_declining'].mean():.3f}")
print(f"Sealed June frame (May->Jun): {len(X_june):,} rows, {meta_june['client_hash_id'].nunique()} clients, "
      f"base rate {meta_june['is_declining'].mean():.3f}")

# ---- Fit on the training frame only; June is never trained on ----
lr_pipe = Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), NUMERIC_COLS),
                                                 ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS)])),
                    ("clf", LogisticRegression(max_iter=2000, random_state=42))])
tree_pipe = Pipeline([("prep", ColumnTransformer([("num", "passthrough", NUMERIC_COLS),
                                                   ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS)])),
                      ("clf", DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=42))])
lr_pipe.fit(X_train, meta_train["is_declining"])
tree_pipe.fit(X_train, meta_train["is_declining"])
TREE_NAMES = NUMERIC_COLS + list(tree_pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_COLS))
print("Both models fit on the April->May frame. June untouched by training.")


def tree_path_strings(pipe, X, feature_names):
    """The <=3 conditions that led each row to its leaf -- the tree's reasoning, verbatim."""
    clf = pipe.named_steps["clf"]
    Xt = pipe.named_steps["prep"].transform(X)
    if sp.issparse(Xt):
        Xt = Xt.toarray()
    Xt = np.asarray(Xt)
    ind = clf.decision_path(Xt)
    leaf = clf.apply(Xt)
    feat, thr = clf.tree_.feature, clf.tree_.threshold
    out = []
    for i in range(Xt.shape[0]):
        conds = []
        for nid in ind.indices[ind.indptr[i]:ind.indptr[i + 1]]:
            if leaf[i] == nid:
                continue
            f, t = feature_names[feat[nid]], thr[nid]
            conds.append(f"{f}<={t:.4g}" if Xt[i, feat[nid]] <= t else f"{f}>{t:.4g}")
        out.append(" AND ".join(conds))
    return out


# ================== SHARED CONSTANTS AND THE FROZEN RULE, DEFINED ONCE ==================
# Defined here so the evidence below and the shipped queue further down use byte-identical
# rule logic -- the ordering choice cannot be justified by one implementation and shipped with another.
POSITION_BINS = [0, 3, 10, 20, 50, np.inf]
POSITION_LABELS = ["top_3", "page_1", "striking_distance", "page_2_3", "deep"]
REVIEW_BUDGET = 50   # documented choice: one reviewer's realistic weekly batch
CV_FOLDS = 10


def precision_at_k(scores, labels, k):
    k = min(k, len(labels))
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order][:k].mean())


def rule_eligible(frame):
    """The three gates carried over unchanged from w04_baseline_score.ipynb."""
    return ((frame["prev_30_impressions"] >= 500)
            & (frame["content_age_days_at_decision"] >= 90)
            & (frame["days_since_last_update_at_decision"] >= 60))


def tier_means_from(frame):
    """Expected CTR per position tier, learned from a frame's OWN eligible rows."""
    tier = pd.cut(frame["prev_30_avg_position"], bins=POSITION_BINS, labels=POSITION_LABELS)
    return (frame.assign(position_tier=tier)[rule_eligible(frame)]
                 .groupby("position_tier", observed=True)["prev_30_ctr"].mean())


def rule_scores_for(frame, tier_means):
    """Frozen rule: score = impressions x ctr_gap where it fires, else 0."""
    tier = pd.cut(frame["prev_30_avg_position"], bins=POSITION_BINS, labels=POSITION_LABELS)
    # .map on a Categorical returns Categorical -- .to_dict() + astype(float) avoids that trap
    expected = tier.map(tier_means.to_dict()).astype(float)
    gap = (expected - frame["prev_30_ctr"]).fillna(0)
    fires = rule_eligible(frame) & (frame["prev_30_ctr"] == 0)
    return np.where(fires, frame["prev_30_impressions"] * gap, 0.0), fires


# ============ EVIDENCE FOR THE ORDERING CHOICE (the numbers quoted in Section 1) ============
# Claim under test: on the April->May frame, the frozen rule orders the HEAD of the queue at least
# as well as either model. Measured the way the queue is actually used -- precision at the review
# budget -- across client-grouped folds, so no client appears in both train and test.
# Each fold re-learns the rule's benchmark from its own training rows, exactly as w06 required.
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

y_train_all = meta_train["is_declining"]
groups_train = meta_train["client_hash_id"]

cv_rows = []
for fold_i, (tr_i, te_i) in enumerate(GroupKFold(n_splits=CV_FOLDS).split(X_train, y_train_all, groups_train), start=1):
    Xtr_f, Xte_f = X_train.iloc[tr_i], X_train.iloc[te_i]
    ytr_f, yte_f = y_train_all.iloc[tr_i], y_train_all.iloc[te_i]

    rule_s, _ = rule_scores_for(Xte_f, tier_means_from(Xtr_f))     # benchmark from TRAIN rows only
    lr_s = clone(lr_pipe).fit(Xtr_f, ytr_f).predict_proba(Xte_f)[:, 1]
    tree_s = clone(tree_pipe).fit(Xtr_f, ytr_f).predict_proba(Xte_f)[:, 1]

    cv_rows.append({"fold": fold_i, "n_test": len(te_i), "base_rate": float(yte_f.mean()),
                    "rule": precision_at_k(rule_s, yte_f.values, REVIEW_BUDGET),
                    "logistic_regression": precision_at_k(lr_s, yte_f.values, REVIEW_BUDGET),
                    "decision_tree": precision_at_k(tree_s, yte_f.values, REVIEW_BUDGET)})

cv = pd.DataFrame(cv_rows)
METHODS = ["rule", "logistic_regression", "decision_tree"]
print(f"\nPrecision@{REVIEW_BUDGET} per client-grouped fold, April->May frame:")
print(cv.round(3).to_string(index=False))

weights = cv["n_test"] / cv["n_test"].sum()
evidence = pd.DataFrame({
    "mean_precision_at_budget": [cv[m].mean() for m in METHODS],
    "weighted_by_fold_size": [float((cv[m] * weights).sum()) for m in METHODS],
    "folds_above_base_rate": [f"{int((cv[m] > cv['base_rate']).sum())}/{CV_FOLDS}" for m in METHODS],
}, index=METHODS).round(3)
print(f"\nMean base rate across folds: {cv['base_rate'].mean():.3f} "
      f"(weighted by fold size: {float((cv['base_rate'] * weights).sum()):.3f})")
print(evidence.to_string())

best = evidence["mean_precision_at_budget"].idxmax()
print(f"\nHighest observed mean precision@{REVIEW_BUDGET}: {best}. This is the evidence behind the")
print("Section 1 ordering choice -- an OBSERVED difference on one frame, not proof of general")
print("superiority. w06_validation_audit.ipynb measured all three methods' precision as")
print("statistically indistinguishable (every pairwise 95% CI includes zero), and the models beat")
print("the rule decisively on recall and F1. The rule wins where this playbook operates: the head")
print("of a short list.")
print(f"\nCaveat on the averaging: folds are very unequal ({cv['n_test'].min():,} to "
      f"{cv['n_test'].max():,} rows) because the panel has few clients, so the unweighted mean lets a")
print("small fold count as much as a large one. Both columns are shown above; the ranking is the")
print("same either way in this run, but check that it still is before quoting a single number.")


# ================== BUILD THE QUEUE (rule orders; models give reason codes) ==================
# Uses the SAME rule helpers the evidence above was computed with.
queue = X_june.copy()
queue["content_hash_id"] = meta_june["content_hash_id"].values
queue["client_hash_id"] = meta_june["client_hash_id"].values
queue["is_declining_june"] = meta_june["is_declining"].values

queue["position_tier"] = pd.cut(queue["prev_30_avg_position"], bins=POSITION_BINS, labels=POSITION_LABELS)
eligible = rule_eligible(queue)
queue["eligible"] = eligible

# The rule infers its benchmark from MAY directly -- the June frame's own prev_30 window,
# fully observable at the 2026-06-01 decision moment. No June information involved.
tier_means = tier_means_from(queue)
print("\nExpected CTR per position tier, learned from May (eligible rows only):")
print(tier_means.round(5).to_string())

queue["expected_ctr_for_tier"] = queue["position_tier"].map(tier_means.to_dict()).astype(float)
queue["ctr_gap"] = (queue["expected_ctr_for_tier"] - queue["prev_30_ctr"]).fillna(0)
queue["rule_score"], rule_fires = rule_scores_for(queue, tier_means)
queue["rule_reason"] = np.where(rule_fires, "CTR_BELOW_POSITION_PEERS", "NO_RULE_FLAG")

queue["lr_decline_probability"] = lr_pipe.predict_proba(X_june)[:, 1]
queue["tree_predicts_decline"] = tree_pipe.predict(X_june)
queue["tree_path"] = tree_path_strings(tree_pipe, X_june, TREE_NAMES)
queue["lr_reason"] = pd.cut(queue["lr_decline_probability"], [-0.01, 0.4, 0.6, 1.01],
                             labels=["LR_LOW_RISK", "LR_MODERATE", "LR_HIGH_RISK"]).astype(str)
queue["tree_reason"] = np.where(queue["tree_predicts_decline"] == 1, "TREE_FLAGS_DECLINE", "TREE_NO_DECLINE")

model_flags = (queue["lr_decline_probability"] >= 0.5) | (queue["tree_predicts_decline"] == 1)
queue["agreement"] = np.select(
    [rule_fires & model_flags, rule_fires & ~model_flags, ~rule_fires & model_flags],
    ["RULE_AND_MODEL_AGREE", "RULE_ONLY", "QUIET_RISK_MODEL_ONLY"], default="NO_FLAG")

# Rule score orders. Rows the rule does not flag all score 0 -- they are NOT ranked by the rule,
# and are placed after the flagged set, broken by impressions descending (a disclosed, neutral
# tiebreak that uses no model output, so the models never influence the ordering).
queue = queue.sort_values(["rule_score", "prev_30_impressions"], ascending=[False, False]).reset_index(drop=True)
queue["queue_rank"] = np.arange(1, len(queue) + 1)
queue["rule_ranked"] = queue["rule_score"] > 0
queue["action"] = np.where(queue["queue_rank"] <= REVIEW_BUDGET, "review", "no_review")

print(f"\nQueue built: {len(queue):,} rows. Rule flags {int(rule_fires.sum()):,}; "
      f"{int(queue['rule_ranked'].sum()):,} carry a non-zero rule score and are genuinely rank-ordered.")
print(f"Top {REVIEW_BUDGET} marked action='review'.\n")
print("Agreement mix inside the review batch:")
print(queue.head(REVIEW_BUDGET)["agreement"].value_counts().to_string())
print("\nTop 10 of the queue:")
display_cols = ["queue_rank", "content_hash_id", "action", "rule_reason", "lr_reason", "tree_reason",
                "position_tier", "prev_30_impressions", "prev_30_ctr", "expected_ctr_for_tier",
                "lr_decline_probability", "agreement"]
queue[display_cols].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_4mo rows: 43647556
date span: (datetime.date(2026, 3, 1), datetime.date(2026, 6, 30))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Training frame (Apr->May): 18,918 rows, 27 clients, base rate 0.543
Sealed June frame (May->Jun): 55,904 rows, 42 clients, base rate 0.671
Both models fit on the April->May frame. June untouched by training.

Precision@50 per client-grouped fold, April->May frame:
 fold  n_test  base_rate  rule  logistic_regression  decision_tree
    1    7115      0.471  0.58                 0.48           0.56
    2    5726      0.634  1.00                 0.70           0.68
    3    2725      0.512  0.50                 0.40           0.74
    4     886      0.694  0.96                 0.88           0.88
    5     607      0.255  0.48                 0.46           0.26
    6     594      0.638  0.62                 0.64           0.74
    7     394      0.495  0.50                 0.68           0.42
    8     291      0.931  0.94                 0.96           1.00
    9     290      0.372  0.60                 0.40           0.42
   10     290      0.617  0.62                 0.84           0.

,queue_rank,content_hash_id,action,rule_reason,lr_reason,tree_reason,position_tier,prev_30_impressions,prev_30_ctr,expected_ctr_for_tier,lr_decline_probability,agreement
0,1,content_ed9c7fe5a778c796,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,15480.0,0.0,0.003808,0.636347,RULE_AND_MODEL_AGREE
1,2,content_e1c8b55bdfecde2a,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,13180.0,0.0,0.003808,0.608132,RULE_AND_MODEL_AGREE
2,3,content_e080fc5170d28a55,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,12357.0,0.0,0.003808,0.923834,RULE_AND_MODEL_AGREE
3,4,content_32446f07f65d452e,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,11790.0,0.0,0.003808,0.621279,RULE_AND_MODEL_AGREE
4,5,content_99fc6465edb0e52c,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_2_3,14657.0,0.0,0.002102,0.665737,RULE_AND_MODEL_AGREE
5,6,content_22fd13cc8de5fc9a,review,CTR_BELOW_POSITION_PEERS,LR_MODERATE,TREE_FLAGS_DECLINE,page_1,7351.0,0.0,0.003808,0.574066,RULE_AND_MODEL_AGREE
6,7,content_c0bfca959627c226,review,CTR_BELOW_POSITION_PEERS,LR_MODERATE,TREE_FLAGS_DECLINE,page_1,6851.0,0.0,0.003808,0.504874,RULE_AND_MODEL_AGREE
7,8,content_17b9e821ceca190e,review,CTR_BELOW_POSITION_PEERS,LR_MODERATE,TREE_FLAGS_DECLINE,page_2_3,11634.0,0.0,0.002102,0.587348,RULE_AND_MODEL_AGREE
8,9,content_76773f1111c808de,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,6133.0,0.0,0.003808,0.708947,RULE_AND_MODEL_AGREE
9,10,content_e84b683e6466e297,review,CTR_BELOW_POSITION_PEERS,LR_HIGH_RISK,TREE_FLAGS_DECLINE,page_1,5943.0,0.0,0.003808,0.923412,RULE_AND_MODEL_AGREE


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended user.** One content strategist or editor deciding which pages to open first in a weekly
review session with a fixed budget of roughly 50 pages. Not an automated pipeline, not a
client-facing report, not a page-quality score.

**Intended use.** Ordering a review queue, and giving each item a reason a human can read and
disagree with. The queue answers *"which pages are worth my attention first this week?"* — it does
not answer *"what is wrong with this page"* or *"what should I change."* Those remain the reviewer's
judgement, informed by opening the page.

**What the evidence supports.** Observed on the April→May frame across 10 client-grouped folds: the
rule's ordering placed more genuinely-declining items in the top 50 than either model's ordering
did (previous section). The
walk-forward check in Section 4 tests whether that ordering still holds on a month the models never
saw. Both are **directional, decision-support** results.

**What it does not support, explicitly:**
- **No causal claim.** Nothing here shows that reviewing or editing a flagged page prevents a
  decline. That needs a controlled comparison — pages held back from review — which does not exist
  in this data. The queue is predictive ordering, not a treatment effect.
- **No ranking-factor claim.** `prev_30_avg_position` and `prev_30_ctr` being useful *predictors* of
  a later impression drop says nothing about what Google's ranking system rewards.
- **No per-client guarantee.** Fold base rates on the April→May frame ranged from 0.255 to 0.931.
  Portfolio-level ordering quality does not transfer automatically to any individual client.

**Where it stops being valid:**

| Boundary | Why |
|---|---|
| Outside the eligibility gates | Items under 500 impressions, under 90 days old, or updated within 60 days are never rule-ranked. Their position in the export is a disclosed impressions tiebreak, not a recommendation. |
| Beyond roughly the top 100 | The rule flags a limited set; past that the queue is tied at score 0 and carries no ordering information. |
| A different month | Section 4 measures how much the base rate moved between May and June. A queue built on a stale benchmark inherits that drift. |
| Clients with little history | Guard (a) requires 60 days of prior data; sparse-history clients are absent, not "healthy". |
| `expected_ctr_for_tier` | Pooled across clients within a tier, so a high-traffic client can dominate the benchmark a low-traffic client is judged against — the same concern `w06` raised about the paper's Finding #3. |

**Known negative result, kept visible.** `w04_signal_audit.ipynb` graded staleness
(`days_since_last_update_at_decision`) **OPPOSITE/MIXED** against CTR: mean CTR rose rather than
fell across the middle staleness buckets, on thin `n` in the older tiers. Staleness therefore
appears here only as an eligibility gate (the 60-day cooldown), never as evidence of decline risk,
and this playbook makes **no decay-or-refresh claim**. That is a reportable finding, not a gap:
the refresh-flag intuition did not survive contact with this portfolio.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --- Scope of the artifact: how much of the queue is actually rule-ordered vs. tiebroken? ---
scope = pd.DataFrame({
    "segment": ["total rows in June frame", "pass all 3 eligibility gates", "rule fires (rank-ordered)",
                 "marked action='review'"],
    "n": [len(queue), int(queue["eligible"].sum()), int(queue["rule_ranked"].sum()), int((queue["action"] == "review").sum())],
})
scope["share_of_frame"] = (scope["n"] / len(queue)).round(3)
print("Scope of the queue:")
print(scope.to_string(index=False))

# --- Reason-code mix, whole queue and review batch ---
print("\nReason-code mix (whole queue):")
print(queue["agreement"].value_counts().to_string())

# --- The stated boundary, verified rather than asserted: below the flagged set the queue is tied ---
n_ranked = int(queue["rule_ranked"].sum())
print(f"\nRows genuinely ordered by the rule: {n_ranked:,}")
print(f"Rows tied at rule_score == 0 (ordering carries NO information): {len(queue) - n_ranked:,}")
if n_ranked < REVIEW_BUDGET:
    print(f"WARNING: fewer rule-flagged rows ({n_ranked}) than the review budget ({REVIEW_BUDGET}) --")
    print("the tail of this week's review batch is impressions-tiebroken, not rule-ranked. Say so in the paper.")
else:
    print(f"The full review budget of {REVIEW_BUDGET} is covered by genuinely rule-ranked rows.")

# --- Per-client concentration: does one client dominate the review batch? ---
batch = queue.head(REVIEW_BUDGET)
conc = batch["client_hash_id"].value_counts()
print(f"\nReview batch spans {conc.shape[0]} distinct clients; largest single client holds "
      f"{conc.iloc[0]} of {REVIEW_BUDGET} slots ({conc.iloc[0] / REVIEW_BUDGET:.0%}).")
print("A batch dominated by one client is a fairness problem for a portfolio-wide reviewer -- flagged, not fixed.")

Scope of the queue:
                     segment     n  share_of_frame
    total rows in June frame 55904           1.000
pass all 3 eligibility gates  7298           0.131
   rule fires (rank-ordered)  1549           0.028
      marked action='review'    50           0.001

Reason-code mix (whole queue):
agreement
QUIET_RISK_MODEL_ONLY    37923
NO_FLAG                  16432
RULE_AND_MODEL_AGREE      1165
RULE_ONLY                  384

Rows genuinely ordered by the rule: 1,549
Rows tied at rule_score == 0 (ordering carries NO information): 54,355
The full review budget of 50 is covered by genuinely rule-ranked rows.

Review batch spans 7 distinct clients; largest single client holds 35 of 50 slots (70%).
A batch dominated by one client is a fairness problem for a portfolio-wide reviewer -- flagged, not fixed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.